# Preliminaries

In [1]:
# Standard library
import os
import pathlib

# Third-party libraries
import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tobler.area_weighted import area_interpolate
from tobler.util import h3fy

%matplotlib inline
%config InlineBackend.figure_format='retina'

## Parent Folders

These should of course be adjusted to reflect the appropriate locations in your disk or wherever

In [2]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)
out_folder = out_folder / 'A'

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder) / 'beaga'

In [3]:
inpath = db_folder / 'crimes_violentos-jan2022-ago2025.csv'
crime = pd.read_csv(inpath, encoding='latin1', sep=';')

crime = crime.loc[
    (crime['Ano Fato'] == 2023)
    & (crime['Município'] == 'BELO HORIZONTE')
    ]

crime = crime.groupby('Bairro - Fato Final').size().reset_index(name='n_crimes')

C:\Users\brand\AppData\Local\Temp\ipykernel_29056\2482840867.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  crime = pd.read_csv(inpath, encoding='latin1', sep=';')


In [4]:
crime.sort_values('n_crimes', ascending=False).head(10)

,Bairro - Fato Final,n_crimes
75,CENTRO,846
282,SANTA EFIGENIA,140
133,FLORESTA,119
247,PADRE EUSTAQUIO,116
371,VILA CLORIS,105
316,SAO LUIZ,97
53,CACHOEIRINHA,96
324,SAVASSI,93
68,CARLOS PRATES,89
199,MANTIQUEIRA,88


In [ ]:
crime.sort_values('n_crimes', ascending=False).head(10)

,Bairro - Fato Final,n_crimes
80,CENTRO,838
285,SANTA EFIGENIA,138
318,SAO LUIZ,126
139,FLORESTA,121
81,CEU AZUL,120
72,CARLOS PRATES,119
326,SAVASSI,119
205,MANTIQUEIRA,113
252,PADRE EUSTAQUIO,103
281,SANTA AMELIA,101


In [5]:
bh = gpd.read_file(db_folder / 'BAIRRO_POPULAR.zip')

In [6]:
crime['bairro_norm'] = (crime['Bairro - Fato Final']
                         .str.strip()
                         .str.upper()
                         .str.normalize('NFKD')
                         .str.encode('ascii', errors='ignore')
                         .str.decode('utf-8'))

bh['bairro_norm'] = (bh['NOME']
                     .str.strip()
                     .str.upper()
                     .str.normalize('NFKD')
                     .str.encode('ascii', errors='ignore')
                     .str.decode('utf-8'))

# Merge on normalized names
merged = bh.merge(crime[['bairro_norm', 'n_crimes']],
                  on='bairro_norm', how='left')

# If needed, drop the helper column
merged = merged.drop(columns='bairro_norm')

In [7]:
inpath  = db_folder / 'ENDERECO.zip'

addresses = (
    gpd.read_file(inpath)
    .dropna(subset=['NUMERO_IMO', 'CEP'])
    .astype({
        'NUMERO_IMO': int,
        'CEP': int,
        })
    .astype({'CEP': str})
    .sort_values(['CEP', 'NUMERO_IMO'])
    .pipe(
        lambda gdf: gdf.loc[gdf.geometry.notnull() & gdf.geometry.is_valid]
        )
    # Column names to lowercase
    .pipe(lambda d: d.rename(str.lower, axis="columns"))
)

In [8]:
addresses_by_borough = (
    merged
    .sjoin(addresses, how="left", predicate="intersects").drop(columns=['index_right'])
    .groupby('NOME')
    .size()
    .to_frame('n_addresses')
)

merged = merged.merge(
    addresses_by_borough,
    left_on='NOME',
    right_index=True,
    how='left'
)


In [9]:
merged['crime_rate'] = merged['n_crimes'].fillna(0) / merged['n_addresses'] * 1_000

In [10]:
merged.crime_rate.describe()

count     496.000000
mean       17.477166
std        68.469106
min         0.000000
25%         2.378834
50%         7.706395
75%        14.447064
max      1000.000000
Name: crime_rate, dtype: float64

In [11]:
merged.sort_values('crime_rate', ascending=False).head(10)

,ID,CODIGO,NOME,AREA_KM2,PERIMETR_M,geometry,n_crimes,n_addresses,crime_rate
80,192.0,817.0,Virgínia,0.05680,1680.485,"POLYGON ((602899.21 7794138.523, 602898.204 77...",1.0,1,1000.000000
305,28.0,633.0,Campus UFMG,3.84979,9534.573,"POLYGON ((607914.304 7800682.125, 607852.543 7...",18.0,19,947.368421
109,9.0,612.0,Baleia,3.62128,11526.160,"POLYGON ((615677.378 7793949.069, 615676.274 7...",7.0,17,411.764706
142,617.0,1972.0,Lagoa da Pampulha,3.13828,18175.488,"POLYGON ((607267.302 7803350.962, 607246.358 7...",5.0,13,384.615385
407,195.0,820.0,Xangri-lá,0.52568,3713.554,"POLYGON ((602698.878 7806267.459, 602702.292 7...",2.0,8,250.000000
282,37.0,642.0,Centro,2.01967,6328.769,"POLYGON ((610062.503 7797318.779, 610103.964 7...",838.0,5527,151.619323
209,454.0,1463.0,Vila Santo Antônio,0.00626,329.277,"POLYGON ((609071.65 7803404.881, 609061.26 780...",5.0,39,128.205128
71,350.0,1358.0,Vila Madre Gertrudes III,0.01757,715.647,"POLYGON ((603617.575 7793811.218, 603649.636 7...",7.0,63,111.111111
395,374.0,1383.0,São Damião,0.21659,2510.802,"POLYGON ((610234.555 7811945.409, 610231.016 7...",4.0,36,111.111111
405,213.0,967.0,Nova América,0.00747,602.718,"POLYGON ((605901.716 7810666.173, 605894.699 7...",2.0,21,95.238095


In [12]:
merged.explore(column='n_crimes', cmap='Reds', legend=True, scheme='naturalbreaks', tiles='CartoDB positron')

c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


In [13]:
merged.explore(column='crime_rate', cmap='Reds', legend=True, scheme='naturalbreaks', tiles='CartoDB positron')

c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


In [61]:
outpath = out_folder / 'crime_by_borough.gpkg'
merged.to_crs(31983).to_file(outpath)

In [62]:
merged

,ID,CODIGO,NOME,AREA_KM2,PERIMETR_M,geometry,n_crimes,n_addresses,crime_rate
0,179.0,804.0,Teixeira Dias,0.73318,4171.623,"POLYGON ((602866.025 7788818.05, 602845.18 778...",6.0,1686,3.558719
1,228.0,1148.0,Diamante,1.83317,9868.308,"POLYGON ((602377.03 7787751.307, 602364.983 77...",30.0,4914,6.105006
2,212.0,926.0,Fernão Dias,0.50501,3182.780,"POLYGON ((613513.975 7801874.842, 613464.236 7...",22.0,1003,21.934197
3,250.0,1173.0,Pousada Santo Antônio,0.39350,3008.336,"POLYGON ((616352.227 7803438.844, 616327.679 7...",15.0,581,25.817556
4,224.0,1144.0,Santa Margarida,0.25388,3264.026,"POLYGON ((603034.537 7791526.279, 603030.72 77...",11.0,809,13.597033
...,...,...,...,...,...,...,...,...,...
491,14.0,619.0,Betânia,1.31105,8511.818,"POLYGON ((605385.667 7791591.075, 605198.568 7...",1.0,8086,0.123671
492,76.0,690.0,Havaí,1.57786,7668.818,"POLYGON ((607025.749 7791809.391, 607026.746 7...",31.0,3949,7.850089
493,370.0,1379.0,Santa Rita de Cássia,0.15789,2081.891,"POLYGON ((610744.847 7792913.456, 610731.588 7...",10.0,1760,5.681818
494,239.0,1161.0,Cinquentenário,0.69409,5050.528,"POLYGON ((606291.176 7791873.658, 606257.046 7...",14.0,1580,8.860759
